# Project 5: Json with xc
Tren Meckel    
November 18th, 2024

#### Import Statements

In [732]:
import json
import csv

### Data Wrangling
Read in the data from cvs and convert into a dictionary, to use as if json file

In [801]:
# Open the file and skip the first two rows
with open("NCAC_XC_2023_Women6k_v2.csv", 'r') as file:
    for i in range(2):  # Skip two rows
        next(file)
    
    # Read the remaining data into a LOD (list of dictionaries)
    reader = csv.DictReader(file)
    data = [row for row in reader]


### Data removal

In [803]:
# Remove rows with missing values (i.e., runners without finishing times)
data = [row for row in data if row["Time"] and row["Time"].strip()]
# Remove unattached runners
data = [runner for runner in data if "Team" in runner and runner["Team"] != "Unattached"]

### Convert name into Firstname , Lastname format

In [805]:
for runner in data:
    full_name = runner["Name"].split()
    runner["LastName"] = full_name[0].capitalize()  # Capitalize to fix any uppercase issues
    runner["FirstName"] = " ".join(full_name[1:]).capitalize()  # Handle multi-part last names
    del runner["Name"]  # Remove the original 'Name' field

### Converting time

In [835]:
for runner in data:
    time_str = runner["Time"]
    
    # Ensure time_str is a string, then split the time into minutes and seconds
    if isinstance(time_str, str):
        try:
            minutes, seconds = time_str.split(":")
            minutes_float = float(minutes)
            seconds_float = float(seconds)
            
            # Convert to total time in minutes
            total_time = minutes_float + (seconds_float / 60)
            
            # Round to one decimal place 
            runner["Time"] = round(total_time, 1)
        except ValueError as e:
            # If there's an issue splitting or converting the time, print an error
            print("ERROR! WRONG! AGAIN!")
            runner["Time"] = None  # Assign None in case of an error
    else:
        runner["Time"] = None  # Assign None if time is not a valid string

### Add Finishing_place & Team_place and Scoring_Place all together 
because they belong together

In [833]:
# Assign initial places to each runner 
for index, runner in enumerate(data):
    runner["Finishing_place"] = index + 1  # Assign finishing place (starting at 1)
    runner["Scoring_place"] = index + 1  # Scoring place starts as finishing place initially

# Create and update team_dict, while also filtering teams to have at most 7 runners
team_dict = {}

# Group runners by team
for runner in data:
    team = runner["Team"]
    if team not in team_dict:
        team_dict[team] = []
    team_dict[team].append(runner)

# Limit teams to a maximum of 7 runners 
for team, runners in team_dict.items():
    if len(runners) > 7:
        # Keep ONLY top 7 runners based on their Scoring_place
        team_dict[team] = sorted(runners, key=lambda x: x["Scoring_place"])[:7]

# Assign 'team_place' within each team, based on the filtered runners
for team, runners in team_dict.items():
    for idx, runner in enumerate(runners):
        runner["Team_place"] = idx + 1  # Assign team place starting from 1

# Update the og data list to reflect the new team sizes 
new_data = []
for runner in data:
    team = runner["Team"]
    if len(team_dict.get(team, [])) > 0:  # If the team still has runners left
        new_data.append(runner)
        team_dict[team].pop(0)  # Remove this runner from the team's list
## team_dict is empty and newdata has the info now

# Update the Scoring_place 
scoring_place_counter = 1
for runner in new_data:  # Use new_data instead of the original data
    runner["Scoring_place"] = scoring_place_counter
    scoring_place_counter += 1

# Final update to the data
data = new_data

In [831]:
# Checking....
print(json.dumps(data[:], indent=4))

[
    {
        "Year": "JR",
        "Team": "Wittenberg",
        "Time": 22.4,
        "LastName": "Khosla,",
        "FirstName": "Sydney",
        "Finishing_place": 1,
        "Scoring_place": 1,
        "Team_place": 1
    },
    {
        "Year": "SO",
        "Team": "Wittenberg",
        "Time": 22.4,
        "LastName": "Webster,",
        "FirstName": "Ella",
        "Finishing_place": 2,
        "Scoring_place": 2,
        "Team_place": 2
    },
    {
        "Year": "JR",
        "Team": "Wittenberg",
        "Time": 22.6,
        "LastName": "Keys,",
        "FirstName": "Megan",
        "Finishing_place": 3,
        "Scoring_place": 3,
        "Team_place": 3
    },
    {
        "Year": "JR",
        "Team": "DePauw",
        "Time": 22.7,
        "LastName": "Porter,",
        "FirstName": "Sophie",
        "Finishing_place": 4,
        "Scoring_place": 4,
        "Team_place": 1
    },
    {
        "Year": "JR",
        "Team": "Wooster",
        "Time": 22.7,
     

### Compute Team Scores

In [829]:
# Initialize an empty list to hold the team scores
team_scores = []

# Create a dictionary to group runners by their team
team_runners = {}

# Group runners by team
for runner in data:
    team = runner["Team"]
    if team not in team_runners:
        team_runners[team] = []
    team_runners[team].append(runner)

# Iterate through the teams and calculate their scores
for team, runners in team_runners.items():
    if len(runners) >= 3:  # Only teams with at least 3 runners are eligible
        
        # Get the top 5 runners based on their scoring place
        top_5_runners = sorted(runners, key=lambda x: x["Scoring_place"] if x["Scoring_place"] is not None else float('inf'))[:5]

        # Calculate the team score by summing the scoring places of the top 5 runners
        score = sum(runner["Scoring_place"] for runner in top_5_runners)

        # Collect the team data
        team_scores.append({
            "team_name": team,
            "score": score,
            "runners": [runner["Scoring_place"] for runner in top_5_runners],
            "other_runners": runners  # the leftovers..
        })

# Sort the teams by their scores 
team_scores.sort(key=lambda x: x["score"])

### Print out the Team Score 

In [827]:
#-----FORMATING------
# Print the final sorted results, including team placement, team name, score, and top 5 runner places
print(f"\n{'Place':<7}{'School':<20}{'Score':<10}{'Runners'}")
for place, team in enumerate(team_scores, start=1):
    # Format the list of top 5 runner scores into a string separated by spaces
    top_5_runners = ' '.join(str(runner) for runner in team["runners"])

    # Get the other runners and format them with parentheses
    other_runners = [runner["Scoring_place"] for runner in team["other_runners"] if runner["Scoring_place"] not in team["runners"]]
    other_runners_str = ' '.join(f"({runner})" for runner in other_runners) if other_runners else ""

    # Print the team's place, name, score, and top 5 runner scores, including the other racers in parentheses
    print(f"{place:<7}{team['team_name']:<20}{team['score']:<10}{top_5_runners} {other_runners_str}")


Place  School              Score     Runners
1      Wittenberg          39        1 2 3 7 26 (30) (33)
2      DePauw              45        4 8 9 11 13 (16) (21)
3      Ohio Wesleyan       98        12 18 20 23 25 (27) (28)
4      Oberlin             109       15 17 19 24 34 (35) (40)
5      Wooster             137       5 14 37 39 42 (46) (49)
6      Kenyon              166       22 29 36 38 41 (43) (45)
7      Denison             205       31 32 44 48 50 (51) (52)
8      Hiram               265       47 53 54 55 56 (57)


### Honors Listing

In [823]:
# Honors for top 5, next 5, and third 5 based on finishing places
honors = {
    'First': [runner for runner in data if runner['Finishing_place'] <= 5],
    'Second': [runner for runner in data if 6 <= runner['Finishing_place'] <= 10],
    'Third': [runner for runner in data if 11 <= runner['Finishing_place'] <= 15]
}

#-----FORMATING------
# All honors 
print("First Team Honors:")
for runner in honors['First']:
    print(f"{runner['Finishing_place']} {runner['FirstName']:>12} {runner['LastName']:>12} {runner['Team']:>20} ")
    
print("Second Team Honors:")
for runner in honors['Second']:
    print(f"{runner['Finishing_place']} {runner['FirstName']:>12} {runner['LastName']:>12} {runner['Team']:>20}")
    
print("Third Team Honors:")
for runner in honors['Third']:
    print(f"{runner['Finishing_place']} {runner['FirstName']:>12} {runner['LastName']:>12} {runner['Team']:>20}")

First Team Honors:
1       Sydney      Khosla,           Wittenberg 
2         Ella     Webster,           Wittenberg 
3        Megan        Keys,           Wittenberg 
4       Sophie      Porter,               DePauw 
5        Dylan   Kretchmar,              Wooster 
Second Team Honors:
6        Alice       Smith,              MadeUpU
7         Emma        Hawk,           Wittenberg
8        Grace      Thomas,               DePauw
9        Grace      Flores,               DePauw
10        Betty      Johson,              MadeUpU
Third Team Honors:
11         Lily     Monnett,               DePauw
12          Zoe        Ward,        Ohio Wesleyan
13     Meredith    Sierpina,               DePauw
14       Athena    Theranos,              Wooster
15         Sage     Reddish,              Oberlin


In [825]:
# Save the modified data to a JSON file
with open('NCAC_Championship_2023_W6k.json', 'w') as outfile:
    json.dump(data, outfile, indent=4)